In [0]:
%python
# TODO: Update these with your VERIFIED credentials from Confluent Cloud
# After verifying in Confluent Cloud console, update these values:

kafka_key = "BBFIDKFOXWBJV52Y"  # Replace with correct key
kafka_secret = "cfltqNcKx6RTn6o01LgbvCGqkMudP0/UWMXYRvytBXEzLz+rd/aNPq59AwmbIuoQ"  # Replace with correct secret
bootstrap_server = "pkc-zm3p0.eu-north-1.aws.confluent.cloud:9092"  # e.g., pkc-xxxxx.region.aws.confluent.cloud:9092

# Once updated, uncomment and run this to clear the old checkpoint:
# dbutils.fs.rm("/Volumes/prism-sentinel-stream/prism_bronze/checkpoints/kafka_bronze", True)

print("⚠️ Update the credentials above before running the stream!")

In [0]:
%python
# IMPORTANT: Run this cell first to delete the old checkpoint
# This removes references to deleted Kafka partitions

checkpoint_path = "/Volumes/prism-sentinel-stream/prism_bronze/checkpoints/kafka_bronze"
checkpoint_path1 = "/Volumes/prism-sentinel-stream/prism_bronze/landing_volume/checkpoints/kafka_bronze"

try:
    dbutils.fs.rm(checkpoint_path, True)
    print(f"✅ Checkpoint deleted: {checkpoint_path}")
    dbutils.fs.rm(checkpoint_path1, True)
    print(f"✅ Checkpoint1111111 deleted: {checkpoint_path1}")
    print("   Stream will start fresh from 'latest' offsets")
    print("\n➡️ Now run the streaming cell below")
except Exception as e:
    print(f"⚠️ Could not delete checkpoint: {e}")
    print("   It may not exist yet, which is fine.")

In [0]:
%python
# --- 01_BRONZE_KAFKA_STREAM.ipynb ---
from pyspark.sql.functions import from_json, col, current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, TimestampType

# 1. KAFKA CONNECTION (Confluent Cloud) - Using credentials from previous cell
kafka_options = {
    "kafka.bootstrap.servers": bootstrap_server,
    "subscribe": "transactions_raw",
    "startingOffsets": "latest",
    "failOnDataLoss": "false",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='{kafka_key}' password='{kafka_secret}';"
}

# 2. DEFINE JSON SCHEMA (matching the transaction structure)
# Update your schema definition
json_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("user_id", StringType(), True),    # Force String
    StructField("amount", StringType(), True),     # Force String
    StructField("country", StringType(), True),
    StructField("counterparty", StringType(), True),
    StructField("timestamp", StringType(), True)
])

# 3. READ STREAM AND PARSE JSON
df_stream = (spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
    .selectExpr("CAST(value AS STRING) as json_string")
    .select(from_json(col("json_string"), json_schema).alias("data"))
    .select(
        col("data.transaction_id"),
        col("data.user_id"),
        col("data.amount"),
        col("data.country"),
        col("data.counterparty"),
        col("data.timestamp"),
        lit("KAFKA_STREAM").alias("ingestion_source")
    ))

# 4. WRITE TO BRONZE TABLE
target_table = "`prism-sentinel-stream`.prism_bronze.transactions_raw"
#checkpoint_path = "/Volumes/prism-sentinel-stream/prism_bronze/checkpoints/kafka_bronze"
checkpoint_path = "/Volumes/prism-sentinel-stream/prism_bronze/landing_volume/checkpoints/kafka_bronze"


print(f"🛰️ Sentinel listening for Kafka events on {bootstrap_server}...")

# Ensure mergeSchema is enabled in the write stream
query = (df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")  # This allows the update to stick
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(target_table))

In [0]:
%python
# Quick way to re-run the stream manually
# This processes any new Kafka messages since last run

from pyspark.sql.functions import from_json, col, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, TimestampType

kafka_options = {
    "kafka.bootstrap.servers": bootstrap_server,
    "subscribePattern": "PRISM_.*",
    "startingOffsets": "latest",
    "failOnDataLoss": "false",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='{kafka_key}' password='{kafka_secret}';"
}

json_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("amount", DecimalType(18, 2), True),
    StructField("country", StringType(), True),
    StructField("counterparty", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])

df_stream = (spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
    .selectExpr("CAST(value AS STRING) as json_string")
    .select(from_json(col("json_string"), json_schema).alias("data"))
    .select(
        col("data.transaction_id"),
        col("data.user_id"),
        col("data.amount"),
        col("data.country"),
        col("data.counterparty"),
        col("data.timestamp"),
        lit("KAFKA_STREAM").alias("ingestion_source")
    ))

target_table = "`prism-sentinel-stream`.prism_bronze.transactions_raw"
checkpoint_path = "/Volumes/prism-sentinel-stream/prism_bronze/checkpoints/kafka_bronze"


##checkpoint_path = "/Volumes/prism-sentinel-stream/prism_bronze/landing_volume/checkpoints/kafka_bronze"

print("🔄 Processing new Kafka messages...")

query = (df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(target_table))

query.awaitTermination()

print("✅ Batch complete! Run again to process more messages.")